In [1]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

In [2]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/Athahar/Advance_RAG_ath/RAG_Main_codes/04_vector_stores')

In [3]:
dotenv_path = project_root.parent / ".env"

load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("EMB_API_KEY"):
    raise ValueError(
        "Please add your EMB_API_KEY to the .env file before running this notebook."
    )

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: c:\Users\Athahar\Advance_RAG_ath\RAG_Main_codes\.env


In [4]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo
Persist directory: c:\Users\Athahar\Advance_RAG_ath\RAG_Main_codes\04_vector_stores\notebooks\chroma_langchain_db


In [ ]:
collection_name = "demo_2"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo_2
Persist directory: c:\Users\Athahar\Advance_RAG_ath\RAG_Main_codes\04_vector_stores\notebooks\chroma_langchain_db


In [6]:
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

Removed the old Chroma directory.


In [7]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.environ["EMB_API_KEY"],
    base_url="https://api.euron.one/api/v1/euri",
)

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Vector store is ready.")

Vector store is ready.


# Add Small helper functions

In [8]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

# Create and Insert Smaller documents

In [9]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [10]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [11]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1. id=6604ba1b-0881-4ae8-98fb-ccf32f04f6c3
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=a12c5591-1ddd-437d-a45a-74f4e24642fe
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=97d3ed26-7359-42fa-bee6-9be90d5811fa
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=f14147d9-a5f9-4314-9d41-70c1b6494ded
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=b5be5252-c4d1-4d12-a4f4-6cbc9e9ecb32
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=5fa1bec8-d855-430e-b766-e65506259c24
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [12]:
documents[0].id

'6604ba1b-0881-4ae8-98fb-ccf32f04f6c3'

In [13]:
# Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
6604ba1b-0881-4ae8-98fb-ccf32f04f6c3
a12c5591-1ddd-437d-a45a-74f4e24642fe
97d3ed26-7359-42fa-bee6-9be90d5811fa
f14147d9-a5f9-4314-9d41-70c1b6494ded
b5be5252-c4d1-4d12-a4f4-6cbc9e9ecb32
5fa1bec8-d855-430e-b766-e65506259c24
3db446cc-e29e-455e-b013-5c209a2eef83
d0465556-0a2c-4597-9c85-f72fff379aa2
bab08868-6226-4b66-b6bc-954b1548bdc3
04545844-a711-4223-b3f0-a5d339f0aa11

Total inserted documents: 10


# Read the stored data back

In [14]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [15]:
raw_records

{'ids': ['6604ba1b-0881-4ae8-98fb-ccf32f04f6c3',
  'a12c5591-1ddd-437d-a45a-74f4e24642fe',
  '97d3ed26-7359-42fa-bee6-9be90d5811fa',
  'f14147d9-a5f9-4314-9d41-70c1b6494ded',
  'b5be5252-c4d1-4d12-a4f4-6cbc9e9ecb32',
  '5fa1bec8-d855-430e-b766-e65506259c24',
  '3db446cc-e29e-455e-b013-5c209a2eef83',
  'd0465556-0a2c-4597-9c85-f72fff379aa2',
  'bab08868-6226-4b66-b6bc-954b1548bdc3',
  '04545844-a711-4223-b3f0-a5d339f0aa11'],
 'embeddings': array([[ 0.00486374,  0.02099609,  0.01657104, ...,  0.00072002,
         -0.0164032 ,  0.0279541 ],
        [-0.0144577 , -0.00542068,  0.03039551, ..., -0.02857971,
         -0.00191212,  0.04272461],
        [ 0.02268982,  0.01535797,  0.04550171, ...,  0.02297974,
          0.00804138, -0.01585388],
        ...,
        [ 0.00804138,  0.02316284,  0.02172852, ..., -0.02069092,
         -0.01786804,  0.01335144],
        [ 0.00883484,  0.06268311,  0.09967041, ..., -0.01579285,
         -0.0137558 ,  0.03677368],
        [ 0.01806641,  0.08551025, 

In [16]:
print(raw_records["embeddings"][0:2, 0:20].shape)

(2, 20)


In [17]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
6604ba1b-0881-4ae8-98fb-ccf32f04f6c3
a12c5591-1ddd-437d-a45a-74f4e24642fe
97d3ed26-7359-42fa-bee6-9be90d5811fa


In [18]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[-3:]
selected_ids

['d0465556-0a2c-4597-9c85-f72fff379aa2',
 'bab08868-6226-4b66-b6bc-954b1548bdc3',
 '04545844-a711-4223-b3f0-a5d339f0aa11']

In [19]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=d0465556-0a2c-4597-9c85-f72fff379aa2
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2. id=bab08868-6226-4b66-b6bc-954b1548bdc3
   topic=Cricket | doc_number=9
   content=Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.
3. id=04545844-a711-4223-b3f0-a5d339f0aa11
   topic=Cricket | doc_number=10
   content=A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.



In [20]:
print(selected_documents)

[Document(id='d0465556-0a2c-4597-9c85-f72fff379aa2', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'), Document(id='bab08868-6226-4b66-b6bc-954b1548bdc3', metadata={'doc_number': 9, 'topic': 'Cricket'}, page_content='Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.'), Document(id='04545844-a711-4223-b3f0-a5d339f0aa11', metadata={'topic': 'Cricket', 'doc_number': 10}, page_content='A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.')]


# Run a Similarity Search

In [21]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [22]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=f14147d9-a5f9-4314-9d41-70c1b6494ded
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=d0465556-0a2c-4597-9c85-f72fff379aa2
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3. id=b5be5252-c4d1-4d12-a4f4-6cbc9e9ecb32
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



In [23]:
search_results

[Document(id='f14147d9-a5f9-4314-9d41-70c1b6494ded', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='d0465556-0a2c-4597-9c85-f72fff379aa2', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
 Document(id='b5be5252-c4d1-4d12-a4f4-6cbc9e9ecb32', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.')]

In [24]:
vector_store.similarity_search_with_score(query=query, k=4)

[(Document(id='f14147d9-a5f9-4314-9d41-70c1b6494ded', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.8066319823265076),
 (Document(id='d0465556-0a2c-4597-9c85-f72fff379aa2', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  0.9389694929122925),
 (Document(id='b5be5252-c4d1-4d12-a4f4-6cbc9e9ecb32', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  1.074934482574463),
 (Document(id='3db446cc-e29e-455e-b013-5c209a2eef83', metadata={'doc_number': 7, 'topic': 'LLM'}, page_content='LLMs generate text by predicting likely next tokens from patterns learned during training.'),
  1.132387638092041)]

# Updating Documents

In [25]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['f14147d9-a5f9-4314-9d41-70c1b6494ded',
 'd0465556-0a2c-4597-9c85-f72fff379aa2']

In [26]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=f14147d9-a5f9-4314-9d41-70c1b6494ded
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=d0465556-0a2c-4597-9c85-f72fff379aa2
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [27]:
print([doc.page_content for doc in documents if doc.id in ids_to_update])

['RAG combines retrieval with generation so the model can answer using external knowledge.', 'Prompt design can improve how clearly an LLM follows instructions and returns useful answers.']


In [28]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
f14147d9-a5f9-4314-9d41-70c1b6494ded
d0465556-0a2c-4597-9c85-f72fff379aa2


In [29]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=f14147d9-a5f9-4314-9d41-70c1b6494ded
metadata={'topic': 'RAG', 'doc_number': 4}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=d0465556-0a2c-4597-9c85-f72fff379aa2
metadata={'doc_number': 8, 'topic': 'LLM'}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [30]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [31]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=f14147d9-a5f9-4314-9d41-70c1b6494ded
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=b5be5252-c4d1-4d12-a4f4-6cbc9e9ecb32
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



# Delete Documents

In [32]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['bab08868-6226-4b66-b6bc-954b1548bdc3',
 '04545844-a711-4223-b3f0-a5d339f0aa11']

In [33]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
bab08868-6226-4b66-b6bc-954b1548bdc3
04545844-a711-4223-b3f0-a5d339f0aa11


In [34]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
6604ba1b-0881-4ae8-98fb-ccf32f04f6c3
a12c5591-1ddd-437d-a45a-74f4e24642fe
97d3ed26-7359-42fa-bee6-9be90d5811fa
f14147d9-a5f9-4314-9d41-70c1b6494ded
b5be5252-c4d1-4d12-a4f4-6cbc9e9ecb32
5fa1bec8-d855-430e-b766-e65506259c24
3db446cc-e29e-455e-b013-5c209a2eef83
d0465556-0a2c-4597-9c85-f72fff379aa2

Deleted ids still present?
bab08868-6226-4b66-b6bc-954b1548bdc3: False
04545844-a711-4223-b3f0-a5d339f0aa11: False


In [35]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
